# 5-class Odor Classifier — LoRA Fine-tuningfruity · sweet · sulfurous · woody · fatty

In [ ]:
%pip install scikit-learn unimol_tools peft huggingface_hub rdkit

In [ ]:
!git clone https://github.com/FufanLu/molecular-foundation-model.git

### Upload embeddings (for labels)

In [ ]:
import os
os.makedirs('molecular-foundation-model/data/processed/embeddings', exist_ok=True)
from google.colab import files
uploaded = files.upload()
for name in uploaded:
    !mv {name} molecular-foundation-model/data/processed/embeddings/

### Load data + encode labels

In [ ]:
import sys
sys.path.append('molecular-foundation-model')

from src.evaluation.similarity import load_embeddings
from src.dataset.load_leffingwell import load_leffingwell
from src.preprocessing.clean_smiles import clean_smiles
from src.classifier.label_encoder import encode_labels, label_distribution, ALL_5_CLASSES

embeddings = load_embeddings()
df = load_leffingwell()
df = clean_smiles(df)
df = encode_labels(df)

print(f"{len(embeddings)} embeddings, {len(df)} molecules")
label_distribution(df)

### Check GPU

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

### Load Uni-Mol model + find attention layer names

In [ ]:
from unimol_tools import UniMolRepr

clf = UniMolRepr(data_type='molecule', remove_hs=False)
model = clf.model

print("All Linear layers in attention modules:")
attn_linears = []
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        if 'attn' in name.lower() or 'attention' in name.lower():
            print(f"  {name}: {module.in_features}->{module.out_features}")
            attn_linears.append(name)
        elif any(k in name.lower() for k in ['query', 'key', 'value', 'q_proj', 'k_proj', 'v_proj']):
            print(f"  {name}: {module.in_features}->{module.out_features}")
            attn_linears.append(name)

print(f"\nFound {len(attn_linears)} attention Linear layers")
print(f"Sample: {attn_linears[:5] if attn_linears else 'NONE FOUND - inspect manually'}")

In [ ]:
# Inspect the model structure more broadly if attn_linears was empty
print("ALL Linear layers in the model:")
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        print(f"  {name}: {module.in_features}->{module.out_features}")

### Apply LoRA adapter
If the attention layer names above contain something like `q_proj`, `k_proj`, `v_proj` — adjust `target_names` below.

In [ ]:
from src.classifier.model import apply_lora

# Adjust these based on the output above
target_names = attn_linears if attn_linears else []
# Fallback: use common Uni-Mol naming patterns
if not target_names:
    target_partial = []
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            target_partial.append(name)
    # Just apply to ALL Linear layers in encoder blocks
    target_names = [n for n in target_partial if 'self_attn' in n or 'fc' not in n]
    if not target_names:
        target_names = [n for n in target_partial if 'encoder' in n and 'layer' in n]

print(f"Target modules: {len(target_names)}")
n_replaced = apply_lora(model, r=4, alpha=8, target_names=target_names)
print(f"LoRA applied to {n_replaced} layers")

### Build classifier

In [ ]:
from src.classifier.model import OdorClassifier

classifier = OdorClassifier(model, hidden_dim=128, num_classes=5, dropout=0.2)
classifier = classifier.cuda() if torch.cuda.is_available() else classifier

trainable = sum(p.numel() for p in classifier.parameters() if p.requires_grad)
total = sum(p.numel() for p in classifier.parameters())
print(f"Trainable: {trainable:,} / {total:,}")

### Preprocess SMILES to Uni-Mol inputs
`UniMolRepr` converts SMILES → 3D coordinates → model inputs internally. We'll use its preprocessing pipeline.

In [ ]:
df_emb = df[df['compound'].isin(list(embeddings.keys()))].reset_index(drop=True)

from sklearn.model_selection import train_test_split
train_idx, test_idx = train_test_split(
    range(len(df_emb)), test_size=0.2, random_state=42
)
print(f"Train: {len(train_idx)}, Test: {len(test_idx)}")

import numpy as np
Y = np.stack(df_emb['y'].values)
train_smiles = df_emb.iloc[train_idx]['smiles'].tolist()
test_smiles = df_emb.iloc[test_idx]['smiles'].tolist()
train_y = torch.tensor(Y[train_idx], dtype=torch.float32)
test_y = torch.tensor(Y[test_idx], dtype=torch.float32)

In [ ]:
# Use UniMolRepr to batch-process SMILES into model inputs
# This re-uses the preprocessing pipeline (RDKit conformer + tokenization)

import gc

def smiles_to_inputs(smiles_list, batch_size=64):
    all_batches = []
    for i in range(0, len(smiles_list), batch_size):
        batch_smiles = smiles_list[i:i+batch_size]
        batch_inputs = clf._prepare_batch(batch_smiles)
        all_batches.append(batch_inputs)
        gc.collect()
    return all_batches

# Try the internal API first
try:
    test_batch = clf._prepare_batch(["CC(=O)O"])
    print("_prepare_batch works!")
    print(f"  keys: {list(test_batch.keys())}")
except AttributeError:
    print("_prepare_batch not found, trying other methods...")
    print(dir(clf)[:20])

In [ ]:
# If _prepare_batch doesn't exist, inspect UniMolRepr internals
import inspect
methods = [m for m in dir(clf) if not m.startswith('_') or m.startswith('__')]
for m in sorted(methods):
    print(m)

### Train LoRA

In [ ]:
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score

device = next(classifier.parameters()).device
optimizer = optim.AdamW(classifier.parameters(), lr=1e-4, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss()

epochs = 30
batch_size = 16

for epoch in range(epochs):
    classifier.train()
    perm = torch.randperm(len(train_smiles))
    total_loss = 0.0

    for i in range(0, len(train_smiles), batch_size):
        idx = perm[i:i+batch_size].tolist()
        batch_smiles = [train_smiles[j] for j in idx]
        
        try:
            batch = clf._prepare_batch(batch_smiles)
        except AttributeError:
            reprs = clf.get_repr(batch_smiles)
            batch = reprs
        
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
                 for k, v in batch.items()}
        batch_y = train_y[idx].to(device)

        optimizer.zero_grad()
        logits = classifier(batch)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Eval
    if epoch % 5 == 0 or epoch == epochs - 1:
        classifier.eval()
        with torch.no_grad():
            all_preds = []
            for j in range(0, len(test_smiles), batch_size):
                test_batch_smiles = test_smiles[j:j+batch_size]
                try:
                    tb = clf._prepare_batch(test_batch_smiles)
                except AttributeError:
                    tb = clf.get_repr(test_batch_smiles)
                tb = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
                      for k, v in tb.items()}
                preds = torch.sigmoid(classifier(tb)).cpu().numpy()
                all_preds.append(preds)
            all_preds = np.concatenate(all_preds, axis=0)
            preds_bin = (all_preds > 0.5).astype(int)
            f1 = f1_score(Y[test_idx][:len(all_preds)], preds_bin, average='macro', zero_division=0)
        print(f"epoch {epoch:3d}  loss {total_loss/max(1,(len(train_smiles)//batch_size)):.4f}  f1_macro {f1:.4f}")
        classifier.train()

In [ ]:
# Save
torch.save(classifier.state_dict(), 'odor_lora.pt')
from google.colab import files
files.download('odor_lora.pt')